# Streaming

Streaming allows you to display parts of the response as they arrive from the model. First, we will collect fragments manually from `model.stream(...)`, and then from the agent via `stream_mode="messages"`.

[Streaming in LangChain](https://docs.langchain.com/oss/python/langchain/streaming)


In [ ]:
# Imports and project path setup only; examples start in the next cell.
from functools import wraps
from pathlib import Path
from time import perf_counter
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from model_config import get_json_model, get_model
from pydantic import BaseModel, Field


In [ ]:
model = get_model()


### 1. Streaming: handle model chunks ourselves

`model.stream(...)` yields `AIMessageChunk` objects. We print each text piece immediately and join the pieces ourselves. A chunk is not necessarily one word or one token.

In [ ]:
def measure_seconds(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            print(f"\n{function.__name__} took {elapsed:.2f} seconds")

    return wrapper

In [ ]:
@measure_seconds
def stream_model_answer(question: str) -> str:
    pieces = []
    print("MODEL: ", end="", flush=True)

    for chunk in model.stream(question):
        text = str(chunk.text)
        print(text, end="", flush=True)
        pieces.append(text)

    answer = "".join(pieces)
    print(f"\nReceived {len(pieces)} chunks; assembled answer: {answer!r}")
    return answer

In [ ]:
stream_model_answer("In an AI chat, what does streaming model output mean? Provide medium lenght and verbose answer please")

### 2. Streaming from a LangChain agent

`stream_mode="messages"` yields `(chunk, metadata)` pairs from the agent's model node. We display each text fragment as it arrives. In lesson 05, `stream_mode="updates"` showed completed model and tool steps instead.


In [ ]:
streaming_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Answer the question from user",
)


@measure_seconds
def stream_agent_answer(question: str) -> str:
    pieces = []
    print("AGENT: ", end="", flush=True)

    for chunk, metadata in streaming_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="messages",
    ):
        if metadata.get("langgraph_node") == "model":
            text = str(chunk.text)
            print(text, end="", flush=True)
            pieces.append(text)

    answer = "".join(pieces)
    print(f"\nReceived {len(pieces)} agent chunks; assembled answer: {answer!r}")
    return answer

In [ ]:
stream_agent_answer("In an AI chat, what does streaming model output mean? Provide medium lenght and verbose answer please")

### 3. JSON arriving in chunks

If the model streams JSON, a single fragment is usually **not a valid JSON document**. We collect all pieces and only run `json.loads` and Pydantic validation after the stream has finished. Schemas and agent state were explained in the previous lesson.


In [ ]:
class StreamSummary(BaseModel):
    topic: str = Field(min_length=1)
    explanation: str = Field(min_length=10)


json_model = get_json_model()
json_prompt = (
    "Explain in one sentence what model response streaming is. "
    "Return only a JSON object with the fields topic and explanation."
)
pieces = []
print("JSON fragments: ", end="", flush=True)
for chunk in json_model.stream(json_prompt):
    piece = str(chunk.text)
    print(piece, end="", flush=True)
    pieces.append(piece)

complete_json = "".join(pieces)
summary = StreamSummary.model_validate(json.loads(complete_json))
print("\nObject after stream completion:", summary)
